# CI/CD for Machine Learning (Continuous Training & Model Deployment)

Applying DevOps principles to Machine Learning requires specialized verification gates:
1. **Data Quality & Schema Gates**: Declarative expectations (Great Expectations pattern) preventing bad data from corrupting models.
2. **Challenger vs Champion Gate**: Automated offline evaluation ensuring new candidates beat current production models.
3. **Slice & Subgroup Fairness Testing**: Guaranteeing models do not degrade on critical customer subgroups or devices.
4. **Automated CI/CD Workflows**: GitHub Actions pipelines orchestrating testing, qualification, and image builds.

In [ ]:
import sys
from pathlib import Path
for p in [Path.cwd() / 'code', Path('07-mlops/ci-cd-for-ml/code')]:
    if p.exists():
        sys.path.insert(0, str(p.resolve()))
import ml_pipeline_ci
print('ml_pipeline_ci loaded')

## 1. Automated Data Validation Gates (Great Expectations Pattern)
Failing fast in CI if input data violates schema constraints, ranges, or allowable categories.

In [ ]:
from ml_pipeline_ci import DataValidator

validator = DataValidator()

training_batch = [
    {"user_id": "u1", "age": 28, "income": 62000, "country": "US"},
    {"user_id": "u2", "age": 45, "income": 95000, "country": "CA"},
    {"user_id": "u3", "age": 34, "income": 71000, "country": "UK"}
]

# Run expectations
validator.expect_column_values_to_not_be_null(training_batch, "user_id")
validator.expect_column_values_to_be_between(training_batch, "age", min_val=18, max_val=100)
validator.expect_column_values_to_be_in_set(training_batch, "country", allowed_set={"US", "CA", "UK", "DE"})

print("Data Validation Summary:")
for res in validator.results:
    status = "PASSED" if res.success else "FAILED"
    print(f" [{status}] {res.expectation_name:38s} on '{res.column}': {res.details}")
print(f"\nAll data validation checks passed? {validator.is_all_passed()}")

## 2. Model Qualification Gate: Challenger vs Champion with Slice Testing
Ensuring that overall accuracy improvements do not conceal regressions on critical sub-populations.

In [ ]:
from ml_pipeline_ci import evaluate_model_gate

test_eval_set = [
    {"device_type": "mobile", "score": 0.8, "label": 1},
    {"device_type": "mobile", "score": 0.2, "label": 0},
    {"device_type": "desktop", "score": 0.9, "label": 1},
    {"device_type": "desktop", "score": 0.3, "label": 0}
]

# Champion model in production
def champ_model(row):
    return 0.7 if row["device_type"] == "mobile" else 0.4

# Challenger model trained in CI
def challenger_model(row):
    return row["score"]  # superior predictor

gate_decision = evaluate_model_gate(
    champion_predict_fn=champ_model,
    challenger_predict_fn=challenger_model,
    test_dataset=test_eval_set,
    slice_key="device_type"
)

print(f"Challenger Model Promoted? {gate_decision.promoted}")
print(f" - Overall Champion   Accuracy: {gate_decision.overall_champ_metric:.2%}")
print(f" - Overall Challenger Accuracy: {gate_decision.overall_challenger_metric:.2%}")
print("\nSlice Breakdown:")
for sl in gate_decision.slice_evaluations:
    print(f" - Slice [{sl.slice_name:7s}]: Champion={sl.champion_metric:.2%} vs Challenger={sl.challenger_metric:.2%} (Pass: {sl.passed})")

## 3. GitHub Actions Continuous Integration Workflow
Declarative pipeline definition for automated testing and container image publication.

In [ ]:
from ml_pipeline_ci import generate_github_actions_workflow

workflow_yaml = generate_github_actions_workflow()
print("--- .github/workflows/ml_ci_cd.yml ---")
print(workflow_yaml[:400] + "\n... [TRUNCATED] ...")